<a href="https://colab.research.google.com/github/Rohan0603/Daemon/blob/master/fine_tuning/notebooks/colab_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Qwen2.5-3B-Instruct for Daemon Desktop Pet (SFT)

This notebook fine-tunes `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` using Supervised Fine-Tuning (SFT) with Unsloth on a free Colab T4 GPU.

**Dataset:** Alpaca-format JSONL with instruction/input/output columns

**Setup:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Connect**
3. Run all cells in order

**Based on:** [unsloth-buddy](https://github.com/TYH-labs/unsloth-buddy) skill

## Cell 1: Install Unsloth & Dependencies

In [1]:
%%capture
!pip install unsloth
# Restart runtime if prompted (Runtime → Restart runtime)

## Cell 2: Verify GPU & Imports

In [2]:
import torch, json
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

from unsloth import FastLanguageModel
import unsloth, trl, transformers, datasets

print(json.dumps({
    "gpu": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "unsloth": unsloth.__version__,
    "trl": trl.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "cuda": torch.version.cuda,
}))
print("GPU ready!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
{"gpu": "Tesla T4", "vram_gb": 15.6, "unsloth": "2026.9.2", "trl": "0.24.0", "transformers": "5.5.0", "datasets": "4.3.0", "cuda": "12.8"}
GPU ready!


## Cell 3: Load Model & Apply LoRA

Loads `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` in 4-bit QLoRA (~2GB VRAM).

In [3]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.9.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Model loaded. Trainable params: None


## Cell 4: Prepare Dataset

**Upload your dataset file** (batch_00000_alpaca_clean.jsonl), then run the cell below.

Expected format: Alpaca JSONL with `instruction`, `input`, `output` columns.

In [11]:
from google.colab import files
import json
from datasets import Dataset

print("Upload batch_00000_alpaca_clean.jsonl:")
uploaded = files.upload()
filename = next(iter(uploaded))

with open(filename, "r", encoding="utf-8") as file:
    raw_data = [json.loads(line) for line in file if line.strip()]

required_fields = {"instruction", "input", "output"}
valid_data = []
empty_output_records = []

for index, record in enumerate(raw_data):
    if not required_fields.issubset(record):
        missing = sorted(required_fields - set(record))
        raise ValueError(f"Record {index} missing fields: {missing}")
    if not isinstance(record["instruction"], str) or not record["instruction"].strip():
        raise ValueError(f"Record {index} has empty instruction")
    if not isinstance(record["input"], str):
        raise ValueError(f"Record {index} input must be a string")
    if not isinstance(record["output"], str):
        raise ValueError(f"Record {index} output must be a string")
    if not record["output"].strip():
        empty_output_records.append(index)
        continue
    valid_data.append(record)

if empty_output_records:
    print(
        f"Skipped {len(empty_output_records)} record(s) with empty output: "
        f"{empty_output_records}"
    )

if len(valid_data) < 10:
    raise ValueError("Dataset must contain at least 10 valid records for a train/evaluation split")

data = valid_data
dataset = Dataset.from_list(data)
split = dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Uploaded: {len(raw_data)} samples")
print(f"Valid: {len(dataset)} | Skipped: {len(empty_output_records)}")
print(f"Train: {len(train_dataset)} | Evaluation: {len(eval_dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Example: {dataset[0]}")

Upload batch_00000_alpaca_clean.jsonl:


Saving batch_00000_alpaca_clean.jsonl to batch_00000_alpaca_clean (2).jsonl
Skipped 1 record(s) with empty output: [208]
Uploaded: 244 samples
Valid: 243 | Skipped: 1
Train: 218 | Evaluation: 25
Columns: ['instruction', 'input', 'output']
Example: {'instruction': 'Mode: desktop_companion\nAPM: 180\nIdle: 0s\nWindow: desktop\nScreen: Terminal test output\nMemory: user_habits: Uses AI for 90% of tasks | user_current_project: Daemon desktop pet\nTrigger: autonomous', 'input': '', 'output': "Oh we're DEFINING things now? Bold of you to start typing before your mouse calms the hell down!"}


## Cell 5: Train with SFT

In [12]:
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    instructions = example["instruction"]
    input_texts = example.get("input", "")
    outputs = example["output"]
    if isinstance(instructions, str):
        instructions = [instructions]
        input_texts = [input_texts]
        outputs = [outputs]
    formatted = []
    for instruction, input_text, output in zip(instructions, input_texts, outputs):
        user_content = instruction + (f"\n{input_text}" if input_text else "")
        messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output},
        ]
        formatted.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return formatted

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_func,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch size = 8
        max_steps = 300,                   # Increase for real training (e.g., 300-500)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        warmup_steps = 10,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

train_result = trainer.train()
print(f"Training complete. Steps: {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/243 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 243 | Num Epochs = 10 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
1,0.633430
2,0.560822
3,0.739125
4,0.650573
5,0.738407
6,0.676733
7,0.642348
8,0.563185
9,0.682886
10,0.721492


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.


Training complete. Steps: 300
Training loss: 0.2492


## Cell 6: Save LoRA Adapters

In [15]:
import json
import os
from datetime import datetime, timezone

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Evaluate only when trainer and eval data were created in the same Cell 5 run.
eval_loss = None
if getattr(trainer, "eval_dataset", None) is not None:
    try:
        eval_metrics = trainer.evaluate()
        eval_loss = eval_metrics.get("eval_loss")
    except (KeyError, ValueError) as error:
        print(
            "Warning: evaluation dataset is incompatible with this trainer instance; "
            f"eval_loss is unavailable ({error}). Rerun Cells 4-5 for evaluation."
        )
else:
    print("Warning: no evaluation dataset attached; eval_loss is unavailable.")

manifest = {
    "base_model": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "dataset": filename,
    "split_seed": 3407,
    "train_examples": len(train_dataset) if "train_dataset" in globals() else None,
    "eval_examples": len(eval_dataset) if "eval_dataset" in globals() else None,
    "max_steps": trainer.args.max_steps,
    "global_step": trainer.state.global_step,
    "training_loss": train_result.training_loss,
    "eval_loss": eval_loss,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
with open("lora_model/artifact_manifest.json", "w", encoding="utf-8") as manifest_file:
    json.dump(manifest, manifest_file, indent=2)

print("Adapters and artifact manifest saved to lora_model/")
for saved_filename in sorted(os.listdir("lora_model")):
    size = os.path.getsize(os.path.join("lora_model", saved_filename))
    print(f"  {saved_filename}: {size / 1e6:.1f} MB")

Adapters and artifact manifest saved to lora_model/
  README.md: 0.0 MB
  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 59.9 MB
  artifact_manifest.json: 0.0 MB
  chat_template.jinja: 0.0 MB
  tokenizer.json: 11.4 MB
  tokenizer_config.json: 0.0 MB


## Cell 7: Test Inference

Test with a Daemon-style prompt to verify the model learned the personality.

In [16]:
import os
from unsloth import FastLanguageModel

assert os.path.isfile("lora_model/adapter_config.json"), (
    "Run Cell 6 first: lora_model/adapter_config.json is missing."
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="lora_model",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
# Avoid the inherited model max_length warning; max_new_tokens controls output size.
model.generation_config.max_length = None


def generate_response(user_content, max_new_tokens=128):
    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def check_response(prompt, response):
    lowered = response.lower()
    generic_phrases = (
        "certainly!",
        "i understand",
        "let's break down",
        "this indicates",
        "this setup is designed",
        "i will primarily",
        "as qwen",
        "i don't have direct access",
        "here's what happens",
    )
    prompt_echo = any(
        line.strip().lower() in lowered
        for line in prompt.splitlines()
        if line.strip()
    )
    generic_explanation = any(
        phrase in lowered for phrase in generic_phrases
    )
    terminal_punctuation = (".", "!", "?", "\"", "'", ")", "`")
    likely_truncated = (
        len(response) >= 500
        and not response.endswith(terminal_punctuation)
    )
    return {
        "non_empty": bool(response),
        "prompt_echo": prompt_echo,
        "generic_explanation": generic_explanation,
        "likely_truncated": likely_truncated,
        "characters": len(response),
    }


test_prompts = [
    "Mode: desktop_companion\nAPM: 120\nIdle: 30s\nWindow: vscode\nScreen: VS Code editor with Python source\nTrigger: autonomous",
    "Mode: desktop_companion\nAPM: 0\nIdle: 180s\nWindow: vscode\nScreen: unchanged Python source\nTrigger: autonomous",
    "Mode: desktop_companion\nAPM: 20\nIdle: 2s\nWindow: vscode\nScreen: Python test failure visible\nTrigger: user",
]

warnings = 0
for index, prompt in enumerate(test_prompts, start=1):
    response = generate_response(prompt)
    checks = check_response(prompt, response)
    print(f"\n--- Test {index} ---\n{response}\nChecks: {checks}")
    if any((checks["prompt_echo"], checks["generic_explanation"], checks["likely_truncated"])):
        warnings += 1
        print("WARNING: manual review needed; response is generic or incomplete.")

print(f"\nMechanical check complete: adapter loaded, responses generated, warnings={warnings}/{len(test_prompts)}.")
print("This is not a pass/fail quality score; review persona and behavior manually.")

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


--- Test 1 ---
Bro you're hammering away at nothing, those keys aren't gonna write themselves, you chaotic gremlin!
Checks: {'non_empty': True, 'prompt_echo': False, 'generic_explanation': False, 'likely_truncated': False, 'characters': 100}

--- Test 2 ---
Tests? Tests where? The whitespace on your keyboard is giving me anxiety!
Checks: {'non_empty': True, 'prompt_echo': False, 'generic_explanation': False, 'likely_truncated': False, 'characters': 73}

--- Test 3 ---
Tests are running, that's more red text than I can handle at once! Somebody tell me which one of you messed up!
Checks: {'non_empty': True, 'prompt_echo': False, 'generic_explanation': False, 'likely_truncated': False, 'characters': 111}

Mechanical check complete: adapter loaded, responses generated, warnings=0/3.
This is not a pass/fail quality score; review persona and behavior manually.


## Cell 8 (Optional): Export to GGUF

Export to GGUF format for use with Ollama, LM Studio, or llama.cpp.

In [18]:
EXPORT_GGUF = False

if EXPORT_GGUF:
    model.save_pretrained_gguf(
        "model_gguf",
        tokenizer,
        quantization_method="q4_k_m",
    )
    print("GGUF exported!")

    from google.colab import files
    import glob
    import os

    gguf_files = glob.glob("model_gguf_gguf/*.gguf")
    if not gguf_files:
        raise FileNotFoundError(
            "No GGUF file found in model_gguf_gguf/ after export"
        )

    for filename in gguf_files:
        size_gb = os.path.getsize(filename) / 1e9
        print(f"Downloading {filename} ({size_gb:.2f} GB)")
        files.download(filename)

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.




Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 12539.03it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)




Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:01<01:01, 61.22s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:32<00:00, 46.37s/it]


Unsloth: Merge process complete. Saved to `/content/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf_gguf/Qwen2.5-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.un

## Cell 9 (Optional): Push to Hugging Face Hub

In [ ]:
# Uncomment and set your HF token:
# HF_TOKEN = "hf_YOUR_TOKEN_HERE"
# model.push_to_hub("your-username/qwen2.5-3b-finetuned", token=HF_TOKEN)
# tokenizer.push_to_hub("your-username/qwen2.5-3b-finetuned", token=HF_TOKEN)

## Download Adapters

Download the `lora_model/` folder from the Colab file browser (left panel → folder icon → right-click → Download).